**Testing Deploy System**

**Import Library**

In [1]:
import requests
import pandas as pd
import tensorflow as tf
import json
import base64

In [2]:
data = pd.read_csv("data\heart.csv")

In [3]:
data

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


**Endpoint Docker Lokal**

In [7]:
inputs = {
    "Age": 22,
    "Sex": "M",
    "ChestPainType": "ATA",
    "RestingBP": 140,
    "Cholesterol": 289,
    "FastingBS": 0,
    "RestingECG": "Normal",
    "MaxHR": 172,
    "ExerciseAngina": "N",
    "Oldpeak": 0.0,
    "ST_Slope": "Up"
}

# === 2. Konversi ke JSON TensorFlow Serving ===
def prepare_json(inputs: dict):
    feature_spec = dict()

    for key, value in inputs.items():
        if isinstance(value, float):
            feature_spec[key] = tf.train.Feature(float_list=tf.train.FloatList(value=[value]))
        elif isinstance(value, int):
            feature_spec[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))
        elif isinstance(value, str):
            feature_spec[key] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[value.encode()]))

    example = tf.train.Example(features=tf.train.Features(feature=feature_spec)).SerializeToString()

    return json.dumps({
        "signature_name": "serving_default",
        "instances": [
            {"examples": {"b64": base64.b64encode(example).decode()}}
        ]
    })

# === 3. Buat JSON Data ===
json_data = prepare_json(inputs)

# === 4. Endpoint Docker Lokal ===
# Pastikan model serving berjalan, contoh:
# docker run -p 8501:8501 -p 8500:8500 --name heart-failure-prediction \
#   -v "E:/project/MlOps_tugas1/saved_model:/models/heart-failure-prediction" \
#   -e MODEL_NAME=heart-failure-prediction \
#   tensorflow/serving
endpoint = "http://localhost:8501/v1/models/Heart-Failure-Prediction:predict"

# === 5. Kirim Request ke Model ===
response = requests.post(endpoint, data=json_data)
print("Response status:", response.status_code)

if response.status_code == 200:
    result_json = response.json()
    prediction = result_json.get("predictions")
    if prediction:
        prediction_value = prediction[0][0]
        result = "No Heart Disease" if prediction_value < 0.5 else "Heart Disease"
        print(f"✅ Prediction Value: {prediction_value:.4f}")
        print(f"🩺 Result: {result}")
    else:
        print("⚠️ Error: No predictions found in response.")
else:
    print("❌ Request failed:", response.text)


Response status: 200
✅ Prediction Value: 0.0014
🩺 Result: No Heart Disease
